<a href="https://colab.research.google.com/github/ekaesha/hirag-ontology/blob/main/hirag_ontology_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# HiRAG-Ontology: Multi-Agent Knowledge Graph Construction

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ekaesha/hirag-ontology/blob/main/hirag_ontology_colab.ipynb)

**Bachelor's Thesis — HSE DSBA 2026**  
Authors: Eva Karimova, Alexey Popov

This notebook demonstrates the full multi-agent pipeline:
- A1: Knowledge extraction from clinical guidelines
- A2: Ontological entity typing
- A3: Hybrid entity deduplication
- A4: Consistency validation
- A5: Missing relation inference
- A6: Knowledge graph update
- Evaluation: Hybrid RRF retrieval vs baselines

**Dataset:** Minzdrav oncological clinical guidelines (78 documents)

In [1]:
# Install dependencies
!pip install sentence-transformers rank-bm25 networkx openai python-dotenv -q

# Clone repository
!git clone https://github.com/ekaesha/hirag-ontology.git
%cd hirag-ontology

Cloning into 'hirag-ontology'...
remote: Enumerating objects: 169, done.
remote: Counting objects: 100% (169/169), done.
remote: Compressing objects: 100% (81/81), done.
remote: Total 169 (delta 100), reused 151 (delta 85), pack-reused 0 (from 0)
Receiving objects: 100% (169/169), 3.14 MiB | 9.97 MiB/s, done.
Resolving deltas: 100% (100/100), done.
/content/hirag-ontology


In [2]:
# Set API key
import os
from google.colab import userdata

try:
    os.environ['DEEPSEEK_API_KEY'] = userdata.get('DEEPSEEK_API_KEY')
    print('API key loaded from Colab Secrets')
except:
    os.environ['DEEPSEEK_API_KEY'] = 'your_key_here'
    print('Set your DEEPSEEK_API_KEY manually')

API key loaded from Colab Secrets


In [3]:
# Load pre-built knowledge graph (skip expensive extraction)
# The graph was built from 10 Minzdrav documents
import json
import sys
sys.path.insert(0, '.')

from pipeline.knowledge_graph import KnowledgeGraph

# Download pre-built graph from GitHub releases
import urllib.request
import os

os.makedirs('results', exist_ok=True)

# Use demo graph if no pre-built available
if not os.path.exists('results/knowledge_graph_final.json'):
    print('Building demo knowledge graph...')
    from evaluation.run_eval import create_demo_graph
    kg = create_demo_graph()
    kg.save('results/knowledge_graph_final.json')
else:
    kg = KnowledgeGraph.load('results/knowledge_graph_final.json')

print(f'Graph loaded: {kg.stats()}')

Building demo knowledge graph...
[Ontology] Loaded: 9 classes, 7 properties, 5 axioms ← ontology.json
[Demo] Graph: {'nodes': 25, 'edges': 23}
[KG] Saved: 25 entities, 23 relations
Graph loaded: {'nodes': 25, 'edges': 23}


In [4]:
# Run A3: Deduplication
from pipeline.deduplication import DeduplicationAgent

dedup = DeduplicationAgent(alpha=0.6, threshold=0.85)
stats = dedup.deduplicate(kg)
print(f"Deduplication: removed {stats['entities_before'] - stats['entities_after']} entities ({stats['reduction_pct']}%)")

  [Dedup] Before: 25 entities, 23 edges
  [Dedup] Loading model from cache...
  [Dedup] No embeddings model: We couldn't connect to 'https://huggingface.co' to load the files, and couldn't find them in the cached files.
Check your internet connection or see how to run the library in offline mode at 'https://huggingface.co/docs/transformers/installation#offline-mode'.
  [Dedup] Found 0 duplicate pairs
  [Dedup] After:  25 entities, 23 edges
  [Dedup] Removed: 0 entities (0.0%)
Deduplication: removed 0 entities (0.0%)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [5]:
# Run A4: Validation — compute Cons(G)
from pipeline.validator import ValidationAgent

validator = ValidationAgent()
val = validator.validate(kg)
print(f"Cons(G) = {val['consistency_score']:.3f}")
print(f"Violations: {val['total_violations']}")
print(f"By type: {val['violations_by_type']}")

  [Validation] Checking 25 entities, 23 relations...
  [Validation] Violations found: 9
  [Validation] Cons(G) = 0.922
Cons(G) = 0.922
Violations: 9
By type: {'domain_constraint': 4, 'range_constraint': 5}


In [6]:
# Compute Q(G)
from pipeline.quality import compute_quality

q = compute_quality(kg, val)
print(f"Q(G) = {q['Q']:.4f}")
print(f"Coverage = {q['coverage']:.3f}")
print(f"Consistency = {q['consistency']:.3f}")
print(f"Precision = {q['precision']:.3f}")


  [Quality] Q(G) components:
    Coverage     = 0.778  (λ1=0.3)
    Consistency  = 0.922  (λ2=0.3)
    Precision    = 1.000  (λ3=0.2)
    Redundancy   = 0.000  (λ4=0.2)
    Q(G)         = 0.710
Q(G) = 0.7098
Coverage = 0.778
Consistency = 0.922
Precision = 1.000


In [7]:
# Run hybrid RRF retrieval
from retrieval.retriever import HybridRetriever, RetrievalMode

kg.compute_pagerank()
retriever = HybridRetriever(kg, mode=RetrievalMode.HYBRID_RRF)

query = 'What is the treatment for acute lymphoblastic leukemia?'
entity_ids = retriever.retrieve(query, top_k=10)
context = kg.get_context_for_entities(entity_ids)

print(f'Query: {query}')
print(f'Retrieved {len(entity_ids)} entities:')
for eid in entity_ids[:5]:
    e = kg.entities.get(eid)
    if e:
        print(f'  - {e.label} ({e.entity_type})')

  [Encoder] Loading BERT model from cache...


OSError: We couldn't connect to 'https://huggingface.co' to load the files, and couldn't find them in the cached files.
Check your internet connection or see how to run the library in offline mode at 'https://huggingface.co/docs/transformers/installation#offline-mode'.

In [ ]:
# Visualise knowledge graph with NetworkX
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from collections import Counter

TYPE_COLORS = {
    'Condition':'#534AB7', 'Drug':'#D85A30',
    'Procedure':'#1D9E75', 'LabTest':'#185FA5',
    'AnatomicalStructure':'#378ADD', 'DosageRegimen':'#BA7517',
    'Symptom':'#639922', 'Organization':'#D4537E', 'Other':'#888780'
}

# Build nx graph
G = nx.DiGraph()
for eid, e in kg.entities.items():
    G.add_node(eid, label=e.label, etype=e.entity_type)
for r in kg.relations:
    G.add_edge(r.subject_id, r.object_id, pred=r.predicate)

# Top 25 by degree
degrees = dict(G.degree())
top25 = [n for n, _ in sorted(degrees.items(), key=lambda x: x[1], reverse=True)[:25]]
subG = G.subgraph(top25).copy()
pos = nx.spring_layout(subG, k=2.5, seed=42)

node_colors = [TYPE_COLORS.get(G.nodes[n].get('etype','Other'),'#888') for n in subG]
node_sizes  = [degrees[n] * 18 + 150 for n in subG]
labels      = {n: G.nodes[n].get('label','')[:12] for n in subG}

fig, ax = plt.subplots(figsize=(14, 9))
nx.draw_networkx_edges(subG, pos, alpha=0.4, arrows=True, ax=ax)
nx.draw_networkx_nodes(subG, pos, node_color=node_colors,
                       node_size=node_sizes, alpha=0.9, ax=ax)
nx.draw_networkx_labels(subG, pos, labels, font_size=7,
                        font_color='white', font_weight='bold', ax=ax)

legend = [mpatches.Patch(color=c, label=t) for t, c in TYPE_COLORS.items()
          if any(G.nodes[n].get('etype') == t for n in subG)]
ax.legend(handles=legend, loc='upper left', fontsize=8)
ax.set_title('Knowledge Graph — Top 25 Entities by Degree', fontsize=13, fontweight='bold')
ax.axis('off')
plt.tight_layout()
plt.savefig('results/colab_graph.png', dpi=150, bbox_inches='tight')
plt.show()
print('Graph saved to results/colab_graph.png')